# ERP Preprocessing Visualization — 3DGS Radiance Field Pipeline

**TCC @ UFRGS — ERP 3D Classification**

Visual inspection of the full preprocessing pipeline for the 3D Gaussian Splat
radiance field ERP approach.

| Section | What you will see |
|---------|-------------------|
| 1 | PLY loading and Gaussian property inspection |
| 2 | Radiance field ERP generation step by step |
| 3 | All 8 ERP shell channels (density maps) |
| 4 | EgoNeRF exponential vs linear shell spacing |
| 5 | SWHDC latitude-dependent dilation weights W_n(phi) |
| 6 | Data augmentation effects (rotation, blur, noise) |
| 7 | GaussianERPDataset sample loading |
| 8 | Per-class gallery (one ERP per ModelNet10 class) |
| 9 | Model input sanity check (HSDCNet, SWHDCResNet) |

---
## Imports and Global Style

In [ ]:
import sys
import warnings
from pathlib import Path

# ── project root on sys.path ─────────────────────────────────────────────────
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.ndimage import gaussian_filter
import torch

# ── src imports ───────────────────────────────────────────────────────────────
from src.preprocessing.ply_loader import load_gaussian_ply
from src.preprocessing.radiance_field import (
    compute_centroid,
    build_ray_directions,
    compute_shell_radii,
    precompute_gaussian_params,
    compute_radiance_field_erp,
    gaussian_ply_to_erp,
)
from src.preprocessing.augmentation import (
    augment,
    rotate_erp_3d,
    gaussian_blur_erp,
    gaussian_noise_erp,
)
from src.preprocessing.dataset import (
    GaussianERPDataset,
    MODELNET10_CATEGORIES,
)

warnings.filterwarnings('ignore')

# ── publication style ────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 200,
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.titleweight': 'bold',
    'axes.labelsize': 9,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
})

# ── output directory for saved figures ───────────────────────────────────────
FIG_DIR = PROJECT_ROOT / 'experiments' / 'figures' / 'preprocessing'
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f'Figures will be saved to: {FIG_DIR}')

---
## Configuration

In [ ]:
# ── paths ─────────────────────────────────────────────────────────────────────
DATA_ROOT  = PROJECT_ROOT / 'gs_data' / 'modelsplat' / 'modelsplat_ply'
CLASS_NAME = 'bathtub'   # any ModelNet10 class

# ERP parameters matching CLAUDE.md defaults
ERP_H, ERP_W = 128, 256  # reduced for notebook speed; production uses 256x512
N_SHELLS = 8
CUTOFF_SIGMA = 3.0
BATCH_SIZE = 128

MN10_CLASSES = list(MODELNET10_CATEGORIES)

# ── find sample ───────────────────────────────────────────────────────────────
def find_sample(category: str, split: str = 'train', index: int = 0):
    """Return PLY path for a given category/split/index."""
    split_dir = DATA_ROOT / category / split
    if not split_dir.exists():
        return None
    objects = sorted(split_dir.iterdir())
    if index >= len(objects):
        return None
    ply = objects[index] / 'point_cloud.ply'
    return ply if ply.exists() else None


PLY_PATH = find_sample(CLASS_NAME, split='train', index=0)
if PLY_PATH is None:
    raise FileNotFoundError(
        f"No PLY found for class '{CLASS_NAME}' under {DATA_ROOT}.\n"
        f"Download ModelSplat data first: python scripts/download_modelsplat.py"
    )

print(f'Class      : {CLASS_NAME}')
print(f'PLY path   : {PLY_PATH}')
print(f'ERP size   : {ERP_H} x {ERP_W}')
print(f'N_SHELLS   : {N_SHELLS}')

---
## Section 1 — PLY Loading & Gaussian Property Inspection

In [ ]:
gs = load_gaussian_ply(PLY_PATH)

xyz     = gs['xyz']      # (N, 3) float32
opacity = gs['opacity']  # (N,)   float32  sigmoid(raw)
scale   = gs['scale']    # (N, 3) float32  exp(raw)
rgb     = gs['rgb']      # (N, 3) float32  0.5 + SH_C0 * f_dc

centroid = compute_centroid(xyz, opacity)
rel = xyz - centroid
r_dist = np.linalg.norm(rel, axis=1)

print(f"N Gaussians  : {gs['n_gaussians']:,}")
print(f"Centroid     : {centroid.round(4)}")
print(f"Opacity mean : {opacity.mean():.4f}")
print(f"Scale mean   : {scale.mean():.5f}")
print(f"r_dist p5/p50/p95: {np.percentile(r_dist, [5,50,95]).round(4)}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(opacity, bins=80, color='steelblue', edgecolor='none', alpha=0.85)
axes[0].axvline(opacity.mean(), color='crimson', linestyle='--', label=f'mean={opacity.mean():.3f}')
axes[0].set_xlabel('Opacity (sigmoid)')
axes[0].set_ylabel('Count')
axes[0].set_title('Opacity Distribution')
axes[0].legend()

axes[1].hist(r_dist, bins=80, color='mediumseagreen', edgecolor='none', alpha=0.85)
axes[1].axvline(r_dist.mean(), color='darkred', linestyle='--', label=f'mean={r_dist.mean():.3f}')
axes[1].set_xlabel('Distance from centroid')
axes[1].set_ylabel('Count')
axes[1].set_title('Radius Distribution')
axes[1].legend()

for c_idx, (col, lab) in enumerate(zip(['#e74c3c','#2ecc71','#3498db'], ['R','G','B'])):
    axes[2].hist(rgb[:, c_idx], bins=80, color=col, alpha=0.55, edgecolor='none', label=lab)
axes[2].set_xlabel('Channel value (SH DC -> [0,1])')
axes[2].set_ylabel('Count')
axes[2].set_title('RGB Channel Distributions')
axes[2].legend()

fig.suptitle(f'Gaussian Properties — {CLASS_NAME}', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(FIG_DIR / '1_gaussian_properties.pdf')
plt.show()

---
## Section 2 — Radiance Field ERP Generation (Step by Step)

In [ ]:
import time

# Step 1: precompute Gaussian parameters
gs_precomp = precompute_gaussian_params(gs, centroid)

# Step 2: EgoNeRF exponential shell radii
shell_radii = compute_shell_radii(gs_precomp['r_dist'], n_shells=N_SHELLS)

# Step 3: ERP ray directions
ray_dirs = build_ray_directions(ERP_H, ERP_W)   # (H*W, 3)

# Step 4: evaluate radiance field
t0 = time.time()
erp = compute_radiance_field_erp(
    gs_precomp=gs_precomp,
    centroid=centroid,
    ray_dirs=ray_dirs,
    shell_radii=shell_radii,
    H=ERP_H,
    W=ERP_W,
    cutoff_sigma=CUTOFF_SIGMA,
    batch_size=BATCH_SIZE,
)
elapsed = time.time() - t0

print(f"ERP shape : {erp.shape}   dtype={erp.dtype}")
print(f"ERP min/max: [{erp.min():.4f}, {erp.max():.4f}]")
print(f"Elapsed  : {elapsed:.1f}s")
print(f"Shell radii (EgoNeRF exp.): {shell_radii.round(4)}")

---
## Section 3 — All 8 ERP Shell Channels

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 7),
                          gridspec_kw={'hspace': 0.45, 'wspace': 0.3})

for s, ax in enumerate(axes.flat):
    d = gaussian_filter(erp[s], sigma=1)
    im = ax.imshow(d, cmap='hot', aspect='auto', interpolation='bilinear')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
    ax.set_title(f'Shell {s+1}  r={shell_radii[s]:.3f}', fontsize=9, pad=4)
    ax.set_xlabel('Azimuth')
    ax.set_ylabel('Elevation')

fig.suptitle(
    f'Radiance Field ERP — all {N_SHELLS} shells  |  {CLASS_NAME}\n'
    f'(EgoNeRF exp. spacing, cutoff={CUTOFF_SIGMA}sigma, H={ERP_H}, W={ERP_W})',
    fontsize=12, y=1.01,
)
fig.savefig(FIG_DIR / '3_all_shells.pdf')
plt.show()

---
## Section 4 — EgoNeRF Exponential vs Linear Shell Spacing

The production pipeline uses EgoNeRF exponential spacing (Choi et al., CVPR 2023, §3.2):

$$r_s = r_{\text{near}} \cdot \left(\frac{r_{\text{far}}}{r_{\text{near}}}\right)^{s/(N-1)}$$

Exponential spacing allocates denser shells near the inner boundary where the
object surface typically lies.  The `modelsplat_visualization.ipynb` notebook
uses simpler linear spacing for illustration, but training always uses exponential.

In [ ]:
r_near = float(np.percentile(r_dist, 5))
r_far  = float(np.percentile(r_dist, 95))

s_idx = np.arange(N_SHELLS)
radii_exp = r_near * (r_far / r_near) ** (s_idx / max(N_SHELLS - 1, 1))
radii_lin = np.linspace(r_near, r_far, N_SHELLS)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(r_dist, bins=100, color='steelblue', alpha=0.7, label='Gaussians', density=True)
for i, r_s in enumerate(radii_exp):
    ax.axvline(r_s, color='#d73027', linestyle='--', linewidth=1.2, alpha=0.8,
               label='EgoNeRF exp.' if i == 0 else '_')
for i, r_s in enumerate(radii_lin):
    ax.axvline(r_s, color='#2166ac', linestyle=':', linewidth=1.2, alpha=0.8,
               label='Linear' if i == 0 else '_')
ax.set_xlabel('Distance from centroid')
ax.set_ylabel('Density')
ax.set_title('Shell Spacing Comparison')
ax.legend()

ax = axes[1]
ax.plot(s_idx + 1, radii_exp, 'o--', color='#d73027', linewidth=1.8, label='EgoNeRF exp. (production)')
ax.plot(s_idx + 1, radii_lin, 's:', color='#2166ac', linewidth=1.8, label='Linear (visualization)')
ax.set_xlabel('Shell index')
ax.set_ylabel('Shell radius')
ax.set_title('Shell Radii vs Index')
ax.legend()

fig.suptitle(
    f'Shell Spacing — EgoNeRF Exponential vs Linear  |  {CLASS_NAME}',
    fontsize=12, y=1.02,
)
plt.tight_layout()
fig.savefig(FIG_DIR / '4_shell_spacing.pdf')
plt.show()

print(f"Exponential radii: {radii_exp.round(4)}")
print(f"Linear radii     : {radii_lin.round(4)}")

---
## Section 5 — SWHDC Latitude-Dependent Dilation Weights W_n(φ)

The SWHDC block corrects ERP distortion by using **different dilation rates at
different latitudes**.  The weight assigned to dilation rate n at row y is:

$$R_\phi = \min\!\left(N,\, \frac{1}{\sin\phi(y)}\right) \quad \text{(SWHDC Eq. 3)}$$

with linear interpolation between the two nearest integer dilation rates (Eq. 4).
N = 4 covers 96.85% of the sphere's surface area.

In [ ]:
N_DILATION = 4

phi_rows = np.pi * (np.arange(ERP_H) + 0.5) / ERP_H   # (H,) in (0, pi)
sin_phi  = np.sin(phi_rows).clip(min=1e-6)
R_phi    = np.minimum(N_DILATION, 1.0 / sin_phi)       # (H,) Eq. 3

W = np.zeros((N_DILATION, ERP_H), dtype=np.float64)    # (4, H)
for row in range(ERP_H):
    r = R_phi[row]
    r_lo = int(np.floor(r))
    r_hi = int(np.ceil(r))
    r_lo = np.clip(r_lo, 1, N_DILATION)
    r_hi = np.clip(r_hi, 1, N_DILATION)
    if r_lo == r_hi:
        W[r_lo - 1, row] = 1.0
    else:
        W[r_hi - 1, row] = r - r_lo
        W[r_lo - 1, row] = r_hi - r

assert np.allclose(W.sum(axis=0), 1.0), 'Weights do not sum to 1'

lat_deg = 90.0 - phi_rows * 180.0 / np.pi
colors_dil = ['#2166ac', '#4dac26', '#d7191c', '#7b3294']
labels_dil = [f'Dilation r={n}' for n in range(1, N_DILATION + 1)]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), gridspec_kw={'wspace': 0.42})

# Panel 1: R_phi vs latitude
axes[0].plot(lat_deg, R_phi, 'k-', lw=2)
for n in range(1, N_DILATION + 1):
    axes[0].axhline(n, color=colors_dil[n-1], ls='--', lw=0.9, alpha=0.7)
axes[0].set_xlabel('Latitude (degrees)')
axes[0].set_ylabel('R_phi  (effective dilation rate)')
axes[0].set_title('R_phi = min(N, 1/sin phi)  [SWHDC Eq. 3]', pad=6)
axes[0].set_xlim(-90, 90)
axes[0].set_xticks(np.arange(-90, 91, 30))
axes[0].invert_xaxis()

# Panel 2: W_n stacked area
W_plot = W[:, ::-1].T
lat_flip = lat_deg[::-1]
axes[1].stackplot(lat_flip, W_plot.T, labels=labels_dil, colors=colors_dil, alpha=0.8)
axes[1].set_xlabel('Latitude (degrees)')
axes[1].set_ylabel('Weight W_n(phi)')
axes[1].set_title('Latitude weights W_n(phi)  [SWHDC Eq. 4]', pad=6)
axes[1].set_xlim(90, -90)
axes[1].set_xticks(np.arange(-90, 91, 30))
axes[1].legend(loc='upper right', fontsize=8)

# Panel 3: dominant dilation rate per ERP row (heatmap)
dominant_rate = R_phi.reshape(-1, 1) * np.ones((1, ERP_W))
im = axes[2].imshow(dominant_rate, cmap='plasma', aspect='auto',
                    origin='upper', vmin=1, vmax=N_DILATION,
                    extent=[-180, 180, -90, 90])
axes[2].set_xlabel('Longitude (degrees)')
axes[2].set_ylabel('Latitude (degrees)')
axes[2].set_title('Effective dilation R_phi per ERP row', pad=6)
cbar = fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.02)
cbar.set_ticks([1, 2, 3, 4])
cbar.set_label('R_phi (dilation rate)')

fig.suptitle('SWHDC Latitude-Dependent Dilation Weights', fontsize=12, y=1.02)
fig.savefig(FIG_DIR / '5_swhdc_latitude_weights.pdf')
plt.show()
print(f'N=4 covers {100 * (R_phi < 4).mean():.2f}% of rows (approx 96.85% spherical area)')

---
## Section 6 — Data Augmentation Effects

Three augmentation primitives, each applied with **15% probability** during training
(HSDC paper §III-A / SWHDC paper §IV-A):

1. **3D rotation** — rotates the spherical signal by remapping pixel directions
   through the inverse rotation matrix (x,y ∈ [0°,15°], z ∈ [0°,45°])
2. **Gaussian blur** — per-channel, σ ∈ [0.1, 2.0]
3. **Gaussian noise** — per-channel, mean ∈ [0, 0.001], σ ∈ [0, 0.03]

All three primitives work channel-agnostically on the (N_shells, H, W) ERP tensor.

In [ ]:
rng_aug = np.random.default_rng(0)

erp_rotated = rotate_erp_3d(erp, angle_x_deg=10.0, angle_y_deg=10.0, angle_z_deg=35.0)
erp_blurred = gaussian_blur_erp(erp, sigma=1.8)
erp_noisy   = gaussian_noise_erp(erp, mean=0.0, std=0.025, rng=rng_aug)
erp_all_aug = augment(erp, prob=1.0, rng=np.random.default_rng(7))  # all three at once

versions = [
    ('Original', erp),
    ('3-D rotation\n(x=10 deg, y=10 deg, z=35 deg)', erp_rotated),
    ('Gaussian blur\n(sigma = 1.8)', erp_blurred),
    ('Gaussian noise\n(sigma = 0.025)', erp_noisy),
    ('All three combined\n(prob=1.0, seed=7)', erp_all_aug),
]

# Show shell 0 (innermost) and shell N_SHELLS//2 (middle) side by side
show_shells = [0, N_SHELLS // 2]

fig, axes = plt.subplots(len(show_shells), len(versions), figsize=(18, 5 * len(show_shells)),
                          gridspec_kw={'hspace': 0.5, 'wspace': 0.3})

for row, s in enumerate(show_shells):
    for col, (title, erp_v) in enumerate(versions):
        ax = axes[row, col]
        d = gaussian_filter(erp_v[s].astype(np.float64), sigma=1)
        vmax = erp_v[s].max() if erp_v[s].max() > 1e-8 else 1.0
        im = ax.imshow(d, cmap='hot', aspect='auto',
                       interpolation='bilinear', vmin=0, vmax=vmax)
        if row == 0:
            ax.set_title(title, fontsize=8.5, pad=4)
        ax.set_ylabel(f'Shell {s+1}' if col == 0 else '')
        ax.set_xlabel('Azimuth')

fig.suptitle(f'Data Augmentation Effects — {CLASS_NAME}', fontsize=12, y=1.02)
fig.savefig(FIG_DIR / '6_augmentation_effects.pdf')
plt.show()
print(f'All three augmentation functions work on (C={N_SHELLS}, H={ERP_H}, W={ERP_W}) tensors.')

In [ ]:
# ── Difference maps: augmented - original ─────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), gridspec_kw={'wspace': 0.35})

diff_labels = [
    ('Rotation diff', erp_rotated[0] - erp[0]),
    ('Blur diff',     erp_blurred[0] - erp[0]),
    ('Noise diff',    erp_noisy[0]   - erp[0]),
    ('All-aug diff',  erp_all_aug[0] - erp[0]),
]
for ax, (lbl, diff) in zip(axes, diff_labels):
    vmax = max(abs(diff).max(), 1e-6)
    im = ax.imshow(diff, cmap='RdBu_r', aspect='auto', origin='upper',
                   vmin=-vmax, vmax=vmax)
    ax.set_title(lbl, fontsize=9, pad=4)
    ax.set_xlabel('Azimuth')
    ax.set_ylabel('Elevation')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

fig.suptitle('Augmentation Difference Maps  (shell 1, augmented - original)',
             fontsize=11, y=1.02)
fig.savefig(FIG_DIR / '6b_augmentation_diffs.pdf')
plt.show()

---
## Section 7 — GaussianERPDataset Sample Loading

Demonstrates the Dataset interface used during training.
Requires `gs_data/modelsplat/modelsplat_ply/` to exist.

In [ ]:
if not DATA_ROOT.exists():
    print('DATA_ROOT not found — skipping GaussianERPDataset demo.')
else:
    ds = GaussianERPDataset(
        data_root=DATA_ROOT,
        categories=MODELNET10_CATEGORIES,
        split='train',
        n_shells=N_SHELLS,
        H=ERP_H,
        W=ERP_W,
        cutoff_sigma=CUTOFF_SIGMA,
        batch_size_rf=BATCH_SIZE,
        augment_train=False,   # no augmentation for visualization
        seed=42,
    )
    print(f'Dataset size (train split): {len(ds)}')

    # Load the first sample
    tensor, label = ds[0]
    print(f'Sample tensor shape : {tensor.shape}')
    print(f'Sample label        : {label}  ({MODELNET10_CATEGORIES[label]})')
    print(f'Tensor dtype        : {tensor.dtype}')
    print(f'Tensor min/max      : [{tensor.min():.4f}, {tensor.max():.4f}]')

    # Quick visualization of the sample
    fig, axes = plt.subplots(2, 4, figsize=(16, 6))
    for s, ax in enumerate(axes.flat):
        d = gaussian_filter(tensor[s].numpy(), sigma=1)
        im = ax.imshow(d, cmap='hot', aspect='auto')
        plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
        ax.set_title(f'Shell {s+1}', fontsize=9)
        ax.axis('off')
    fig.suptitle(
        f'GaussianERPDataset sample — class {MODELNET10_CATEGORIES[label]}',
        fontsize=12, fontweight='bold',
    )
    plt.tight_layout()
    fig.savefig(FIG_DIR / '7_dataset_sample.pdf')
    plt.show()

---
## Section 8 — Per-Class Gallery

One representative ERP per ModelNet10 class (middle shell, density map).
Run this section only if `DATA_ROOT` is available.

In [ ]:
if not DATA_ROOT.exists():
    print('DATA_ROOT not found — skipping gallery.')
else:
    mid_shell = N_SHELLS // 2

    fig, axes = plt.subplots(2, 5, figsize=(18, 7.5),
                              gridspec_kw={'hspace': 0.55, 'wspace': 0.3})

    for ax, cls in zip(axes.flat, MN10_CLASSES):
        ply_path = find_sample(cls, split='train', index=0)
        if ply_path is None:
            ax.text(0.5, 0.5, f'{cls}\n(not found)',
                    ha='center', va='center', transform=ax.transAxes)
            ax.axis('off')
            continue
        try:
            g    = load_gaussian_ply(ply_path)
            c    = compute_centroid(g['xyz'], g['opacity'])
            gp   = precompute_gaussian_params(g, c)
            sr   = compute_shell_radii(gp['r_dist'], n_shells=N_SHELLS)
            rd   = build_ray_directions(ERP_H, ERP_W)
            erp_cls = compute_radiance_field_erp(
                gp, c, rd, sr, ERP_H, ERP_W, CUTOFF_SIGMA, BATCH_SIZE
            )
            d = gaussian_filter(erp_cls[mid_shell].astype(np.float64), sigma=1)
            ax.imshow(d, cmap='hot', aspect='auto', interpolation='bilinear')
            ax.set_title(f'{cls}\nshell {mid_shell+1}/{N_SHELLS}', fontsize=9, pad=4)
        except Exception as exc:
            ax.text(0.5, 0.5, f'{cls}\nerror:\n{exc}',
                    ha='center', va='center', transform=ax.transAxes,
                    fontsize=7, color='red')
        ax.axis('off')

    fig.suptitle(
        f'Per-Class Gallery — ModelNet10  (middle shell {mid_shell+1} density)',
        fontsize=12, y=1.01,
    )
    fig.savefig(FIG_DIR / '8_per_class_gallery.pdf')
    plt.show()

---
## Section 9 — Model Input Sanity Check

Verify that HSDCNet and SWHDCResNet accept the (N_shells, H, W) ERP tensor
and produce logits of the expected shape.

In [ ]:
from src.models.backbones.resnet_hsdc import HSDCNet, SWHDCResNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Build a random (N_shells, 256, 512) ERP batch (2 samples, production resolution)
PROD_H, PROD_W = 256, 512
batch = torch.randn(2, N_SHELLS, PROD_H, PROD_W)

# HSDCNet — ResNet-34 + HSDC
model_hsdc = HSDCNet(in_channels=N_SHELLS, num_classes=10).to(device)
with torch.no_grad():
    logits_hsdc = model_hsdc(batch.to(device))

n_params_hsdc = sum(p.numel() for p in model_hsdc.parameters() if p.requires_grad)
print(f'HSDCNet  | input {tuple(batch.shape)} -> output {tuple(logits_hsdc.shape)} | '
      f'{n_params_hsdc/1e6:.2f}M params')

# SWHDCResNet — ResNet-50 + SWHDC
model_swhdc = SWHDCResNet(in_channels=N_SHELLS, num_classes=10).to(device)
with torch.no_grad():
    logits_swhdc = model_swhdc(batch.to(device))

n_params_swhdc = sum(p.numel() for p in model_swhdc.parameters() if p.requires_grad)
print(f'SWHDCResNet | input {tuple(batch.shape)} -> output {tuple(logits_swhdc.shape)} | '
      f'{n_params_swhdc/1e6:.2f}M params')

print()
print('Reference parameter counts:')
print('  ResNet-34 + HSDC (paper, 12-ch): 5.3M')
print('  ResNet-50 + SWHDC (paper, 1-ch): 25.5M')
print('  (our 8-channel models have slightly different counts due to in_channels change)')

---
## Section 10 — Pipeline Summary Figure

A single figure showing the 3DGS preprocessing pipeline:
PLY file -> Gaussian properties -> Shell assignment -> ERP channels -> Model input.

In [ ]:
fig = plt.figure(figsize=(20, 4))
gs_fig = gridspec.GridSpec(1, 5, figure=fig, wspace=0.35)

SHELL_SHOW = [0, N_SHELLS // 4, N_SHELLS // 2, N_SHELLS - 1]

# Panel 1: opacity distribution
ax0 = fig.add_subplot(gs_fig[0])
ax0.hist(opacity, bins=60, color='steelblue', alpha=0.85, edgecolor='none')
ax0.axvline(opacity.mean(), color='crimson', linestyle='--', linewidth=1.5)
ax0.set_title('(1) Opacity\ndistribution', fontsize=9)
ax0.set_xlabel('sigmoid(raw)', fontsize=7)

# Panels 2-5: 4 representative ERP shells
titles_p = ['(2) Shell 1\n(inner)', '(3) Shell 3', '(4) Shell 5', '(5) Shell 8\n(outer)']
for col, (s, ttl) in enumerate(zip(SHELL_SHOW, titles_p), 1):
    ax = fig.add_subplot(gs_fig[col])
    d = gaussian_filter(erp[s].astype(np.float64), sigma=1)
    ax.imshow(d, cmap='hot', aspect='auto')
    ax.set_title(ttl, fontsize=9)
    ax.axis('off')

fig.suptitle(
    f'3DGS ERP Preprocessing Pipeline Summary  |  {CLASS_NAME}',
    fontsize=12, y=1.05,
)
fig.savefig(FIG_DIR / '10_pipeline_summary.pdf')
plt.show()
print(f'\nAll figures saved to: {FIG_DIR}')